<a href="https://colab.research.google.com/github/ernanovi/end-to-end-erp-bi/blob/main/API_to_BigQuery_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install library pendukung untuk BigQuery (kalau belum ada)
!pip install --quiet pandas-gbq google-cloud-bigquery

# 2. Import library yang kita butuhkan
import pandas as pd
from google.colab import auth

# 3. Autentikasi akun Google kamu
auth.authenticate_user()
print("Autentikasi Berhasil! Siap lanjut ke langkah berikutnya.")

Autentikasi Berhasil! Siap lanjut ke langkah berikutnya.


In [2]:
import requests
import pandas as pd
from google.colab import auth

# 1. Tarik Data dari API ERP (Simulasi menggunakan API Publik Data Transaksi/Produk)
url = "https://dummyjson.com/products?limit=50"  # Mengambil data produk/transaksi
response = requests.get(url)
data = response.json()

# 2. Ubah data JSON dari API menjadi bentuk Tabel (DataFrame)
# ambil list produknya dan rapikan kolomnya ala data ERP
products_list = data['products']
df_erp_api = pd.DataFrame(products_list)

# Pilih dan sesuaikan kolom yang diperlukan
df_erp_clean = df_erp_api[['id', 'title', 'category', 'price', 'stock', 'rating']].copy()
df_erp_clean.rename(columns={
    'id': 'product_id',
    'title': 'product_name',
    'price': 'unit_price'
}, inplace=True)

# membuat kolom tiruan untuk simulasi data penjuala
df_erp_clean['quantity_sold'] = 3  # Contoh simulasi jumlah terjual
df_erp_clean['total_revenue'] = df_erp_clean['unit_price'] * df_erp_clean['quantity_sold']

print("Data dari API ERP berhasil ditarik dan dibersihkan!")
display(df_erp_clean.head())

# Isi dgn project id bigquery
project_id = 'belajar-dwh-1'  # <-- Ganti dengan Project ID BigQuery kamu
dataset_id = 'erp_simulation'
table_id = 'api_erp_sales'

destination_table = f"{dataset_id}.{table_id}"

# 3. Alirkan (Load) Data dari API langsung ke BigQuery (DWH)
df_erp_clean.to_gbq(
    destination_table,
    project_id=project_id,
    if_exists='replace'
)

print(f"Data dari API ERP berhasil dialirkan ke BigQuery pada tabel: {destination_table}")

Data dari API ERP berhasil ditarik dan dibersihkan!


,product_id,product_name,category,unit_price,stock,rating,quantity_sold,total_revenue
0,1,Essence Mascara Lash Princess,beauty,9.99,99,2.56,3,29.97
1,2,Eyeshadow Palette with Mirror,beauty,19.99,34,2.86,3,59.97
2,3,Powder Canister,beauty,14.99,89,4.64,3,44.97
3,4,Red Lipstick,beauty,12.99,91,4.36,3,38.97
4,5,Red Nail Polish,beauty,8.99,79,4.32,3,26.97


/tmp/ipykernel_2940/2512459909.py:38: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_erp_clean.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 9118.05it/s]

Data dari API ERP berhasil dialirkan ke BigQuery pada tabel: erp_simulation.api_erp_sales
